In [ ]:
import json
import time
from pathlib import Path

import anthropic
import pandas as pd
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request
from dotenv import load_dotenv

data_dir = Path.cwd().parents[1] / 'data'
load_dotenv(dotenv_path=data_dir.parent / ".env")

client = anthropic.Anthropic()

JUDGE_MODEL = "claude-opus-5"
JUDGE_MAX_TOKENS = 180  # 4 ints + a 1-sentence reasoning fits comfortably
VALIDATION_SAMPLE_SIZE = 75
VALIDATION_RANDOM_STATE = 42
AGREEMENT_WITHIN1_THRESHOLD = 0.80
AGREEMENT_CORR_THRESHOLD = 0.70
SCORE_GAP_THRESHOLD = 1  # points, on the 1-10 scale

model_outputs_df = pd.read_parquet(data_dir / "step2_model_outputs.parquet")
print(f"Loaded {len(model_outputs_df)} queries with model outputs")

In [ ]:
# Step 3a: judge prompt + schema. Independent, per-output scoring (never comparative/pairwise).
JUDGE_PROMPT_TEMPLATE = """You are an expert evaluator scoring how well an AI assistant's response answers a user's query.

Query:
{query}

Response:
{response}

Score this response on three dimensions, each a 1-10 integer (1 = completely fails, 10 = excellent):
- correctness: is the information/reasoning factually and logically correct?
- completeness: does the response fully address everything the query asked for?
- quality: is the response well-written, clear, and appropriately detailed (not padded, not truncated)?

Also give an overall_score (1-10) reflecting your holistic judgment of whether this response
is "good enough" - weight correctness most heavily, then completeness, then quality.

Give your reasoning as ONE concise sentence (max 20 words) justifying your scores."""

JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "overall_score": {"type": "integer"},
        "correctness_score": {"type": "integer"},
        "completeness_score": {"type": "integer"},
        "quality_score": {"type": "integer"},
        "reasoning": {"type": "string"},
    },
    "required": ["overall_score", "correctness_score", "completeness_score", "quality_score", "reasoning"],
    "additionalProperties": False,
}

In [ ]:
# Build and submit the judge batch: one request per (query_id, model_type) pair with a real output
def build_judge_batch_requests(df):
    requests = []
    for row in df.itertuples():
        for slot, output_text in (("small", row.small_model_output), ("large", row.large_model_output)):
            if output_text is None or (isinstance(output_text, float) and pd.isna(output_text)):
                continue
            prompt = JUDGE_PROMPT_TEMPLATE.format(query=row.query, response=output_text)
            requests.append(
                Request(
                    custom_id=f"{row.query_id}__{slot}_judge",
                    params=MessageCreateParamsNonStreaming(
                        model=JUDGE_MODEL,
                        max_tokens=JUDGE_MAX_TOKENS,
                        # Opus 5 runs thinking on by default; thinking and output text
                        # share max_tokens, which can exhaust the budget before any JSON
                        # is written. Disable it for this short structured-output task.
                        thinking={"type": "disabled"},
                        messages=[{"role": "user", "content": prompt}],
                        output_config={"format": {"type": "json_schema", "schema": JUDGE_SCHEMA}},
                    ),
                )
            )
    return requests

judge_batch = client.messages.batches.create(requests=build_judge_batch_requests(model_outputs_df))
print(f"Submitted judge batch {judge_batch.id}, status={judge_batch.processing_status}")

In [ ]:
def wait_for_batch(batch_id, poll_seconds=60):
    while True:
        b = client.messages.batches.retrieve(batch_id)
        if b.processing_status == "ended":
            return b
        print(f"status={b.processing_status}, request_counts={b.request_counts}")
        time.sleep(poll_seconds)

judge_batch = wait_for_batch(judge_batch.id)
print(f"Judge batch ended. request_counts={judge_batch.request_counts}")

In [ ]:
# Parse into LONG format (one row per query x model) -- this is what the validation sample draws from.
# "succeeded but no text" (e.g. thinking ate the whole max_tokens budget) is treated
# as a real failure, not silently left null, so it can be retried below.
def clip_score(value):
    if value is None:
        return None
    try:
        return max(1, min(10, int(value)))
    except (TypeError, ValueError):
        return None

def extract_text(result):
    return next((b.text for b in result.result.message.content if b.type == "text"), None)

rows = []
failed = []
for result in client.messages.batches.results(judge_batch.id):
    query_id, rest = result.custom_id.split("__")
    model_type = rest.replace("_judge", "")
    text = extract_text(result) if result.result.type == "succeeded" else None
    if text is not None:
        scores = json.loads(text)
        rows.append({
            "query_id": query_id,
            "model_type": model_type,
            "overall_score": clip_score(scores.get("overall_score")),
            "correctness_score": clip_score(scores.get("correctness_score")),
            "completeness_score": clip_score(scores.get("completeness_score")),
            "quality_score": clip_score(scores.get("quality_score")),
            "reasoning": scores.get("reasoning"),
        })
    else:
        rows.append({
            "query_id": query_id, "model_type": model_type,
            "overall_score": None, "correctness_score": None,
            "completeness_score": None, "quality_score": None, "reasoning": None,
        })
        error = result.result.type if result.result.type != "succeeded" else f"empty_text:{result.result.message.stop_reason}"
        failed.append({"query_id": query_id, "model_type": model_type, "error": error})

judge_long_df = pd.DataFrame(rows)
judge_long_df.to_parquet(data_dir / "step3_judge_scores_long.parquet", index=False)
print(f"Saved {len(judge_long_df)} judge scores. {len(failed)} failed to score.")

In [ ]:
# Retry failed (query_id, model_type) judge pairs, up to 2 rounds.
retries = 0
while failed and retries < 2:
    retries += 1
    print(f"Retry round {retries}: resubmitting {len(failed)} failed judge pairs...")

    retry_requests = []
    for row in pd.DataFrame(failed).itertuples():
        output_text = model_outputs_df.set_index("query_id").loc[row.query_id, f"{row.model_type}_model_output"]
        query_text = model_outputs_df.set_index("query_id").loc[row.query_id, "query"]
        prompt = JUDGE_PROMPT_TEMPLATE.format(query=query_text, response=output_text)
        retry_requests.append(
            Request(
                custom_id=f"{row.query_id}__{row.model_type}_judge",
                params=MessageCreateParamsNonStreaming(
                    model=JUDGE_MODEL,
                    max_tokens=JUDGE_MAX_TOKENS,
                    thinking={"type": "disabled"},
                    messages=[{"role": "user", "content": prompt}],
                    output_config={"format": {"type": "json_schema", "schema": JUDGE_SCHEMA}},
                ),
            )
        )

    retry_batch = client.messages.batches.create(requests=retry_requests)
    retry_batch = wait_for_batch(retry_batch.id)

    judge_long_df = pd.read_parquet(data_dir / "step3_judge_scores_long.parquet")
    still_failed = []
    for result in client.messages.batches.results(retry_batch.id):
        query_id, rest = result.custom_id.split("__")
        model_type = rest.replace("_judge", "")
        text = extract_text(result) if result.result.type == "succeeded" else None
        mask = (judge_long_df["query_id"] == query_id) & (judge_long_df["model_type"] == model_type)
        if text is not None:
            scores = json.loads(text)
            judge_long_df.loc[mask, "overall_score"] = clip_score(scores.get("overall_score"))
            judge_long_df.loc[mask, "correctness_score"] = clip_score(scores.get("correctness_score"))
            judge_long_df.loc[mask, "completeness_score"] = clip_score(scores.get("completeness_score"))
            judge_long_df.loc[mask, "quality_score"] = clip_score(scores.get("quality_score"))
            judge_long_df.loc[mask, "reasoning"] = scores.get("reasoning")
        else:
            error = result.result.type if result.result.type != "succeeded" else f"empty_text:{result.result.message.stop_reason}"
            still_failed.append({"query_id": query_id, "model_type": model_type, "error": error})

    judge_long_df.to_parquet(data_dir / "step3_judge_scores_long.parquet", index=False)
    failed = still_failed
    print(f"After retry {retries}: {len(failed)} still failed.")

print(f"Step 3a done. Final failed judge-scoring count: {len(failed)}.")

In [ ]:
# Step 3b: export a BLIND validation sample (judge score hidden) for manual hand-scoring
scored = judge_long_df.dropna(subset=["overall_score"])
sample = scored.sample(n=min(VALIDATION_SAMPLE_SIZE, len(scored)), random_state=VALIDATION_RANDOM_STATE).copy()

query_lookup = model_outputs_df.set_index("query_id")["query"]
output_lookup = model_outputs_df.set_index("query_id")

def get_output(row):
    col = f"{row['model_type']}_model_output"
    return output_lookup.loc[row["query_id"], col]

sample["query"] = sample["query_id"].map(query_lookup)
sample["output"] = sample.apply(get_output, axis=1)
sample = sample[["query_id", "model_type", "query", "output"]].copy()
sample["human_score"] = ""
sample.to_csv(data_dir / "step3_validation_sample.csv", index=False)
print(f"Exported {len(sample)} rows to data/step3_validation_sample.csv for manual scoring.")
print("Fill in the human_score column (1-10, same rubric as the judge prompt above),")
print("save as data/step3_validation_sample_scored.csv, then run the next cell.")

In [ ]:
# Step 3c: compute agreement -- this is a real gate, not decorative. Do NOT proceed to Step 4 if it fails
# without an explicit, documented reason. See rationale below for how this run was actually resolved.
from scipy import stats
from sklearn.metrics import cohen_kappa_score

scored_path = data_dir / "step3_validation_sample_scored.csv"
manual_scored = pd.read_csv(scored_path)
merged = manual_scored.merge(
    judge_long_df[["query_id", "model_type", "overall_score"]],
    on=["query_id", "model_type"],
)
merged = merged.dropna(subset=["human_score", "overall_score"])
merged["diff"] = merged["human_score"] - merged["overall_score"]

exact_match_rate = (merged["diff"] == 0).mean()
within_1_rate = merged["diff"].abs().le(1).mean()
pearson_r, _ = stats.pearsonr(merged["human_score"], merged["overall_score"])
spearman_r, _ = stats.spearmanr(merged["human_score"], merged["overall_score"])
kappa = cohen_kappa_score(merged["human_score"], merged["overall_score"], weights="quadratic")
no_outliers = merged[merged["diff"].abs() < 5]
pearson_no_outliers, _ = stats.pearsonr(no_outliers["human_score"], no_outliers["overall_score"])
gate_passed_strict = bool(within_1_rate >= AGREEMENT_WITHIN1_THRESHOLD and pearson_r >= AGREEMENT_CORR_THRESHOLD)

report = {
    "n_validated": int(len(merged)),
    "exact_match_rate": float(exact_match_rate),
    "within_1_rate": float(within_1_rate),
    "pearson_correlation": float(pearson_r),
    "spearman_correlation": float(spearman_r),
    "quadratic_weighted_kappa": float(kappa),
    "pearson_excluding_2_outliers": float(pearson_no_outliers),
    "within_1_threshold": AGREEMENT_WITHIN1_THRESHOLD,
    "correlation_threshold": AGREEMENT_CORR_THRESHOLD,
    "gate_passed_strict": gate_passed_strict,
}
print(json.dumps(report, indent=2))

# On this run: gate_passed_strict == False (correlation 0.62 vs 0.70 threshold), while
# within_1_rate passed comfortably (85% vs 80%). Spearman and quadratic weighted kappa
# both land in the same ~0.62 range, confirming it isn't a Pearson-specific artifact.
# Excluding the single worst 2 of 75 items (both genuinely ambiguous edge cases -- a
# malformed/incomplete query and a debatable classification puzzle) raises Pearson to
# 0.73. After reviewing this diagnosis, the decision was to proceed with the existing
# judge scores rather than spend additional budget re-scoring, documenting this as a
# known limitation rather than silently overriding a failed gate.
report["decision"] = "PROCEED_WITH_DOCUMENTED_LIMITATION" if not gate_passed_strict else "GATE_PASSED"
report["rationale"] = (
    "Strict correlation gate failed; within-1-point agreement passed. Spearman and "
    "quadratic weighted kappa confirm this isn't a Pearson-specific artifact. Excluding "
    "the 2 worst (genuinely ambiguous) items of 75 raises Pearson to 0.73. Proceeding "
    "with a documented limitation rather than spending more budget to re-score."
) if not gate_passed_strict else "Gate passed on both within-1 agreement and correlation."

(data_dir / "step3_agreement_report.json").write_text(json.dumps(report, indent=2))
gate_passed = True  # proceeding per the documented decision above; see report for the full diagnosis

In [ ]:
# Step 4-5: derive label and save the final Phase 1 deliverable. Only run once gate_passed is True.
assert gate_passed, "Judge/human agreement gate failed -- fix the judge before deriving labels."

judge_wide = judge_long_df.pivot(index="query_id", columns="model_type")
judge_wide.columns = [
    f"{model}_model_score" if stat == "overall_score" else f"{model}_{stat}"
    for stat, model in judge_wide.columns
]
judge_wide = judge_wide.reset_index()

labeled = model_outputs_df.merge(judge_wide, on="query_id", how="inner")
labeled = labeled.dropna(subset=["small_model_score", "large_model_score"])

labeled["label"] = (
    (labeled["large_model_score"] - labeled["small_model_score"]) <= SCORE_GAP_THRESHOLD
).astype(int)

FINAL_COLUMNS = [
    "query_id", "query", "source_category", "query_word_count",
    "small_model_output", "large_model_output",
    "small_model_score", "large_model_score",
    "small_correctness_score", "small_completeness_score", "small_quality_score", "small_reasoning",
    "large_correctness_score", "large_completeness_score", "large_quality_score", "large_reasoning",
    "label",
]
final_df = labeled[FINAL_COLUMNS]
final_df.to_parquet(data_dir / "query_dataset.parquet", index=False)

print(f"Saved data/query_dataset.parquet: {len(final_df)} rows")
print(final_df["label"].value_counts())